In [1]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", 200)
pd.set_option("display.max_rows", 100)

df = pd.read_csv(
    "../data/accepted_2007_to_2018Q4.csv",
    low_memory = False
)

print(f"Loaded {len(df):,} rows and {len(df.columns)} columns")

Loaded 2,260,701 rows and 151 columns


In [2]:
data_dict = pd.read_csv(
    '../data/Lending Club Data Dictionary Approved.csv',  
    encoding='latin-1'
)
print(f"Dictionary has {len(data_dict)} entries")
data_dict.head(20)

Dictionary has 153 entries


,LoanStatNew,Description,Unnamed: 2,Unnamed: 3,Unnamed: 4,Unnamed: 5,Unnamed: 6,Unnamed: 7,Unnamed: 8,Unnamed: 9,Unnamed: 10
0,acc_now_delinq,The number of accounts on which the borrower i...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,acc_open_past_24mths,Number of trades opened in past 24 months.,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,addr_state,The state provided by the borrower in the loan...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,all_util,Balance to credit limit on all trades,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,annual_inc,The self-reported annual income provided by th...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,annual_inc_joint,The combined self-reported annual income provi...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,application_type,Indicates whether the loan is an individual ap...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,avg_cur_bal,Average current balance of all accounts,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,bc_open_to_buy,Total open to buy on revolving bankcards.,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,bc_util,Ratio of total current balance to high credit/...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [3]:
data_dict = data_dict[["LoanStatNew", "Description"]].dropna(subset = ["LoanStatNew"])
print(f"Clean dictionary: {len(data_dict)} field definitions\n")

print(f"Columns in the dataset:")
for i, col in enumerate(df.columns):
    print(f"{i:3d}: {col}")

Clean dictionary: 151 field definitions

Columns in the dataset:
  0: id
  1: member_id
  2: loan_amnt
  3: funded_amnt
  4: funded_amnt_inv
  5: term
  6: int_rate
  7: installment
  8: grade
  9: sub_grade
 10: emp_title
 11: emp_length
 12: home_ownership
 13: annual_inc
 14: verification_status
 15: issue_d
 16: loan_status
 17: pymnt_plan
 18: url
 19: desc
 20: purpose
 21: title
 22: zip_code
 23: addr_state
 24: dti
 25: delinq_2yrs
 26: earliest_cr_line
 27: fico_range_low
 28: fico_range_high
 29: inq_last_6mths
 30: mths_since_last_delinq
 31: mths_since_last_record
 32: open_acc
 33: pub_rec
 34: revol_bal
 35: revol_util
 36: total_acc
 37: initial_list_status
 38: out_prncp
 39: out_prncp_inv
 40: total_pymnt
 41: total_pymnt_inv
 42: total_rec_prncp
 43: total_rec_int
 44: total_rec_late_fee
 45: recoveries
 46: collection_recovery_fee
 47: last_pymnt_d
 48: last_pymnt_amnt
 49: next_pymnt_d
 50: last_credit_pull_d
 51: last_fico_range_high
 52: last_fico_range_low
 53: 

In [4]:
df = df[df["loan_status"].isin(['Fully Paid', 'Charged Off'])].copy()
df["default"] = (df['loan_status'] == 'Charged Off').astype(int)

df["issue_d"] = pd.to_datetime(df["issue_d"], format = "%b-%Y")
df["issue_year"] = df["issue_d"].dt.year

df["earliest_cr_line"] = pd.to_datetime(df["earliest_cr_line"], format = "%b-%Y")
df['credit_history_months'] = (
    (df['issue_d'] - df['earliest_cr_line']).dt.days / 30.44
).round().astype('Int64')

print(f"Rows: {len(df):,}, default rate: {df['default'].mean():.2%}")

Rows: 1,345,310, default rate: 19.96%


In [5]:
leakage_payment = [
    'pymnt_plan', 'out_prncp', 'out_prncp_inv',
    'total_pymnt', 'total_pymnt_inv', 'total_rec_prncp',
    'total_rec_int', 'total_rec_late_fee', 'recoveries',
    'collection_recovery_fee', 'last_pymnt_d', 'last_pymnt_amnt',
    'next_pymnt_d', 'last_credit_pull_d',
    'last_fico_range_high', 'last_fico_range_low',
]

leakage_hardship = [
    'hardship_flag', 'hardship_type', 'hardship_reason', 'hardship_status',
    'deferral_term', 'hardship_amount', 'hardship_start_date', 'hardship_end_date',
    'payment_plan_start_date', 'hardship_length', 'hardship_dpd', 'hardship_loan_status',
    'orig_projected_additional_accrued_interest', 'hardship_payoff_balance_amount',
    'hardship_last_payment_amount',
]

leakage_settlement = [
    'debt_settlement_flag', 'debt_settlement_flag_date', 'settlement_status',
    'settlement_date', 'settlement_amount', 'settlement_percentage', 'settlement_term',
]

junk = ['id', 'member_id', 'url', 'policy_code']
text_features = ['emp_title', 'desc', 'title']
borderline = ['disbursement_method']
post_processing = ['loan_status', 'issue_d', 'earliest_cr_line']


cols_to_drop = (post_processing + borderline + text_features + junk + leakage_settlement + leakage_hardship + leakage_payment)

print(f"Columns before: {df.shape[1]}")
df_model = df.drop(columns=cols_to_drop)
print(f"Columns after:  {df_model.shape[1]}")
print(f"Dropped:        {df.shape[1] - df_model.shape[1]}")

Columns before: 154
Columns after:  105
Dropped:        49


In [6]:
df_model.to_parquet('../data/loans_clean.parquet', index=False)
print(f"Saved clean dataset: {df_model.shape[0]:,} rows, {df_model.shape[1]} columns")

Saved clean dataset: 1,345,310 rows, 105 columns
